# Tolerance Analysis for a Manufacturing Scenario

When manufacturing a linkage, dimensional variations in link lengths change the
output path. This notebook shows how to:

1. **Sensitivity analysis**: Identify which dimensions most affect the output
2. **Tolerance analysis**: Monte Carlo simulation of manufacturing variations

**Scenario:** A four-bar crank-rocker linkage for a packaging machine. The coupler
point must follow a precise path. We need to determine which dimensions require
tighter tolerances and what the expected path deviation will be.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

from pylinkage.linkage.sensitivity import analyze_sensitivity, analyze_tolerance
from pylinkage.synthesis import fourbar_from_lengths
from pylinkage.visualizer import plot_static_linkage

## 1. Define the Nominal Linkage

A Grashof crank-rocker four-bar with link lengths a=1, b=3, c=3, d=4
(where a is the shortest link and a+d < b+c guarantees Grashof condition).

In [ ]:
linkage = fourbar_from_lengths(
    crank_length=1.0,
    coupler_length=3.0,
    rocker_length=3.0,
    ground_length=4.0,
    iterations=100,
    name="packaging_fourbar",
)

print(f"Joints: {[j.name for j in linkage.components]}")
print(f"Constraints: {linkage.get_num_constraints()}")

In [ ]:
# Simulate and plot the nominal coupler path with mechanism
loci = list(linkage.step(iterations=100))
coupler_path = [(pos[-1][0], pos[-1][1]) for pos in loci if pos[-1][0] is not None]

fig, ax = plt.subplots(figsize=(8, 6))

# Draw mechanism
plot_static_linkage(linkage, ax, loci, show_legend=True, show_labels=True,
                    show_loci=False,
                    title='Nominal four-bar linkage (a=1, b=3, c=3, d=4)')

# Coupler curve
if coupler_path:
    cx, cy = zip(*coupler_path, strict=False)
    ax.plot(cx, cy, 'b-', linewidth=2, label='Nominal coupler curve')

ax.set_xlabel('X (mm)')
ax.set_ylabel('Y (mm)')
ax.legend(fontsize=8)
plt.tight_layout()
plt.show()

## 2. Sensitivity Analysis

Sensitivity analysis perturbs each constraint by a small relative amount (1%)
and measures the resulting change in the output path. Higher sensitivity means
tighter tolerance is needed for that dimension.

In [ ]:
sensitivity = analyze_sensitivity(
    linkage,
    output_joint=linkage.components[-1],  # Coupler joint C
    delta=0.01,                        # 1% perturbation
    include_transmission=True,
)

print("Sensitivity ranking (most to least sensitive):")
print("=" * 50)
for name, sens in sensitivity.sensitivity_ranking:
    print(f"  {name:20s}: {sens:.4f}")

print(f"\nMost sensitive constraint: {sensitivity.most_sensitive}")
if sensitivity.baseline_transmission is not None:
    print(f"Baseline transmission angle: {sensitivity.baseline_transmission:.1f} deg")

In [ ]:
# Bar chart of sensitivities
names = [name for name, _ in sensitivity.sensitivity_ranking]
values = [abs(s) for _, s in sensitivity.sensitivity_ranking]

fig, ax = plt.subplots(figsize=(10, 5))
colors = ['#e74c3c' if v == max(values) else '#3498db' for v in values]
ax.barh(names, values, color=colors)
ax.set_xlabel('Sensitivity coefficient')
ax.set_title('Sensitivity of coupler path to each constraint dimension')
ax.invert_yaxis()
plt.tight_layout()
plt.show()

## 3. Tolerance Allocation

Based on sensitivity results, we assign tighter tolerances to the most sensitive
dimensions. Typical CNC machining tolerances are +/- 0.02 to +/- 0.1 mm.

In [ ]:
# Assign tolerances based on sensitivity ranking
# Tightest tolerance (0.02 mm) for most sensitive, relaxed (0.1 mm) for least
tolerance_map = {}
for _i, (name, sens) in enumerate(sensitivity.sensitivity_ranking):
    if abs(sens) > 0.1:
        tolerance_map[name] = 0.02   # Tight: grinding/precision machining
    elif abs(sens) > 0.01:
        tolerance_map[name] = 0.05   # Medium: standard CNC
    else:
        tolerance_map[name] = 0.10   # Relaxed: rough machining

print("Tolerance allocation:")
print("=" * 50)
for name, tol in tolerance_map.items():
    grade = {0.02: 'TIGHT', 0.05: 'MEDIUM', 0.10: 'RELAXED'}[tol]
    print(f"  {name:20s}: +/- {tol:.2f} mm ({grade})")

## 4. Monte Carlo Tolerance Analysis

Run 1000 Monte Carlo samples: each sample randomly perturbs constraints within
the allocated tolerances and simulates the linkage. The result is a "cloud" of
possible output paths.

In [ ]:
tolerance_result = analyze_tolerance(
    linkage,
    tolerances=tolerance_map,
    output_joint=linkage.components[-1],
    n_samples=1000,
    seed=42,
)

print(f"Mean path deviation:  {tolerance_result.mean_deviation:.4f} mm")
print(f"Max path deviation:   {tolerance_result.max_deviation:.4f} mm")
print(f"Std deviation:        {tolerance_result.std_deviation:.4f} mm")
print(f"Samples generated:    {tolerance_result.output_cloud.shape[0]}")

In [ ]:
# Plot the tolerance cloud
fig, ax = plt.subplots(figsize=(10, 8))
tolerance_result.plot_cloud(ax=ax, alpha=0.05)
ax.set_xlabel('X (mm)')
ax.set_ylabel('Y (mm)')
plt.tight_layout()
plt.show()

## 5. Per-Position Deviation

Some parts of the coupler curve may be more sensitive than others. Plot the
standard deviation as a function of position around the cycle.

In [ ]:
fig, ax = plt.subplots(figsize=(10, 4))
angles = np.linspace(0, 360, len(tolerance_result.position_std))
ax.plot(angles, tolerance_result.position_std, 'b-', linewidth=2)
ax.fill_between(angles, 0, tolerance_result.position_std, alpha=0.2)
ax.set_xlabel('Crank angle (degrees)')
ax.set_ylabel('Position std deviation (mm)')
ax.set_title('Output path uncertainty vs crank angle')
ax.grid(True)
plt.tight_layout()
plt.show()

## 6. What-If: Tighter Tolerances

Compare path deviation with tighter tolerances on the most sensitive dimension.

In [ ]:
# Halve the tolerance on the most sensitive constraint
tight_map = tolerance_map.copy()
most_sensitive = sensitivity.most_sensitive
tight_map[most_sensitive] = tight_map[most_sensitive] / 2

tight_result = analyze_tolerance(
    linkage,
    tolerances=tight_map,
    output_joint=linkage.components[-1],
    n_samples=1000,
    seed=42,
)

print(f"Original max deviation:  {tolerance_result.max_deviation:.4f} mm")
print(f"Tighter max deviation:   {tight_result.max_deviation:.4f} mm")
improvement = (1 - tight_result.max_deviation / tolerance_result.max_deviation) * 100
print(f"Improvement:             {improvement:.1f}%")
print(f"(Tightened {most_sensitive} from +/-{tolerance_map[most_sensitive]:.2f} "
      f"to +/-{tight_map[most_sensitive]:.2f} mm)")

In [ ]:
# Side-by-side comparison
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 7))

tolerance_result.plot_cloud(ax=ax1, alpha=0.05)
ax1.set_title(f'Original tolerances (max dev: {tolerance_result.max_deviation:.3f})')

tight_result.plot_cloud(ax=ax2, alpha=0.05)
ax2.set_title(f'Tighter {most_sensitive} (max dev: {tight_result.max_deviation:.3f})')

plt.tight_layout()
plt.show()

## Summary

1. **Sensitivity analysis** identified the most critical dimensions
2. **Tolerance allocation** assigned tighter tolerances to sensitive dimensions
3. **Monte Carlo simulation** quantified expected path deviation
4. **What-if analysis** showed the benefit of tightening the most critical tolerance

**Key insights:**
- Not all dimensions are equally important — sensitivity analysis lets you focus
  manufacturing effort where it matters most
- Monte Carlo results include per-position statistics, useful for identifying
  regions of the cycle where precision matters most
- The `to_dataframe()` methods export results to pandas for further analysis